In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "b/1.6-viz"
try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from recsys.io.artifact_store import ArtifactStore
from recsys.analysis.distribution import distribution_summary_table
from recsys.viz.plots import (
    bar_counts,
    box_by_group,
    corr_heatmap,
    hist_box,
    long_tail_hist,
    scatter_sample,
)

outputs_root = getattr(CFG, "outputs_dir", Path(os.environ["DATA_ROOT"]) / "outputs")
store = ArtifactStore(outputs_root)

# Use latest available upstream stage (outliers > missing > loaded) and record in manifest
input_stage_used = None
upstream = None
for candidate in ("outliers", "missing", "loaded"):
    if (Path(outputs_root) / candidate / "LATEST").exists():
        upstream = store.load(candidate)
        input_stage_used = candidate
        break

loaded = store.load("loaded")
recipes = upstream["recipes"] if "recipes" in upstream.keys() else loaded["recipes"]
interactions = upstream["interactions"] if "interactions" in upstream.keys() else loaded["interactions"]

print(f"Using input stage: {input_stage_used} ({upstream.version})")
print(f"Recipes: {recipes.shape}, Interactions: {interactions.shape}")

In [ ]:
numeric_cols = [
    "minutes",
    "n_steps",
    "n_ingredients",
    "calories",
    "total_fat_pdv",
    "sugar_pdv",
    "sodium_pdv",
    "protein_pdv",
    "saturated_fat_pdv",
    "carbohydrates_pdv",
]
numeric_cols = [c for c in numeric_cols if c in recipes.columns]

dist_table = distribution_summary_table(recipes, numeric_cols)
display(dist_table)

figures = {}
fig_idx = 1

for col in numeric_cols:
    key_raw = f"fig_06_{fig_idx:02d}_{col}_raw"
    figures[key_raw] = hist_box(recipes[col], title=f"{col} (Raw)", log1p=False)
    fig_idx += 1

    key_log = f"fig_06_{fig_idx:02d}_{col}_log1p"
    figures[key_log] = hist_box(recipes[col], title=f"{col} (log1p)", log1p=True)
    fig_idx += 1

print(f"Generated {len(figures)} numeric distribution figures.")



In [ ]:
# 1. Rating counts (class imbalance, including 0 = unrated)
figures[f"fig_06_{fig_idx:02d}_rating_counts"] = bar_counts(
    interactions["rating"],
    top_n=6,
    title="Interaction Rating Counts (0 = Unrated)",
    xlabel="Count",
    ylabel="Rating",
    horizontal=False,
)
fig_idx += 1

# 2. Top 20 ingredients & tags
ing_exploded = recipes["ingredients"].explode().rename("ingredient")
figures[f"fig_06_{fig_idx:02d}_top20_ingredients"] = bar_counts(
    ing_exploded, top_n=20, title="Top 20 Raw Ingredients"
)
fig_idx += 1

tag_exploded = recipes["tags"].explode().rename("tag")
figures[f"fig_06_{fig_idx:02d}_top20_tags"] = bar_counts(
    tag_exploded, top_n=20, title="Top 20 Recipe Tags"
)
fig_idx += 1

# 3. Long-tail interaction counts & Cold-start statistics
user_counts = interactions.groupby("user_id").size().rename("reviews_per_user")
recipe_counts = interactions.groupby("recipe_id").size().rename("reviews_per_recipe")

pct_single_review_users = round(float((user_counts == 1).mean() * 100), 2)
pct_single_review_recipes = round(float((recipe_counts == 1).mean() * 100), 2)
cold_start_summary = pd.DataFrame(
    [
        {
            "entity": "user",
            "total_unique": int(len(user_counts)),
            "single_review_count": int((user_counts == 1).sum()),
            "single_review_pct": pct_single_review_users,
            "median_reviews": float(user_counts.median()),
            "max_reviews": int(user_counts.max()),
        },
        {
            "entity": "recipe",
            "total_unique": int(len(recipe_counts)),
            "single_review_count": int((recipe_counts == 1).sum()),
            "single_review_pct": pct_single_review_recipes,
            "median_reviews": float(recipe_counts.median()),
            "max_reviews": int(recipe_counts.max()),
        },
    ]
)
display(cold_start_summary)
print(f"Cold-start finding: {pct_single_review_users}% of users have only 1 review.")

figures[f"fig_06_{fig_idx:02d}_user_reviews_long_tail"] = long_tail_hist(
    user_counts, log=True, title="Reviews per User (Long-Tail)", xlabel="Reviews per User"
)
fig_idx += 1

figures[f"fig_06_{fig_idx:02d}_recipe_reviews_long_tail"] = long_tail_hist(
    recipe_counts, log=True, title="Reviews per Recipe (Long-Tail)", xlabel="Reviews per Recipe"
)
fig_idx += 1



In [ ]:
# Exploratory avg_rating (excluding rating == 0; model targets are computed on train-only in 1.9)
rated_agg = (
    interactions.loc[interactions["rating"] > 0]
    .groupby("recipe_id")["rating"]
    .agg(avg_rating="mean", n_ratings="count")
    .reset_index()
)
recipes_viz = recipes.merge(rated_agg, left_on="id", right_on="recipe_id", how="left")

# Scatter plots
figures[f"fig_06_{fig_idx:02d}_scatter_ingredients_minutes"] = scatter_sample(
    recipes_viz, "n_ingredients", "minutes", n=20_000, alpha=0.2, log_y=True,
    title="Preparation Minutes vs Number of Ingredients (log y)"
)
fig_idx += 1

figures[f"fig_06_{fig_idx:02d}_scatter_calories_rating"] = scatter_sample(
    recipes_viz, "calories", "avg_rating", n=20_000, alpha=0.2, log_x=True,
    title="Average Rating vs Calories (log x)"
)
fig_idx += 1

figures[f"fig_06_{fig_idx:02d}_scatter_steps_rating"] = scatter_sample(
    recipes_viz, "n_steps", "avg_rating", n=20_000, alpha=0.2,
    title="Average Rating vs Number of Steps"
)
fig_idx += 1

# Correlation heatmap (Spearman for skewed features)
heatmap_cols = numeric_cols + ["avg_rating", "n_ratings"]
figures[f"fig_06_{fig_idx:02d}_corr_heatmap"] = corr_heatmap(
    recipes_viz, cols=heatmap_cols, method="spearman", title="Spearman Correlation Heatmap"
)
fig_idx += 1

# Box plot of avg_rating by top 10 culinary tags
tag_rating_df = recipes_viz[["tags", "avg_rating"]].dropna().explode("tags")
figures[f"fig_06_{fig_idx:02d}_box_rating_by_tag"] = box_by_group(
    tag_rating_df, value="avg_rating", group="tags", top_n_groups=10,
    title="Average Rating by Top 10 Recipe Tags"
)
fig_idx += 1

print(f"Total figures ready to save: {len(figures)}")



In [ ]:
inputs_list = [upstream] if upstream.stage == "loaded" else [upstream, loaded]

objects_to_save = {
    "distribution_summary": dist_table,
    "cold_start_summary": cold_start_summary,
    **figures,
}

saved = store.save(
    stage="figures",
    objects=objects_to_save,
    inputs=inputs_list,
    params={
        "input_stage_used": input_stage_used,
        "input_version": upstream.version,
        "n_figures": len(figures),
        "scatter_sample_n": 20_000,
        "corr_method": "spearman",
        "dpi": 150,
    },
)

for fig in figures.values():
    plt.close(fig)

print("Saved figures artifact:", saved.path)
print("Version:", saved.version, "| Input stage:", input_stage_used)